# Polytope Lib Tests

In [ ]:
from pycvxset import Polytope
import numpy as np
import matplotlib.pyplot as plt

import os
import mosek

os.environ["MOSEKLM_LICENSE_FILE"] = "27000@192.168.25.63"
mosek.Env().getversion()
mosek.Env().putlicensepath("27000@192.168.25.63")
mosek.Env().expirylicenses()

In [ ]:
from itertools import product
def cartesian_product(*polytopes):
    V_rep = [p.V for p in polytopes]
    Combinations = []
    for veritces in product(*V_rep):
        Combinations.append(np.concatenate(veritces))
    return Polytope(V=np.array(Combinations))

In [ ]:
# Define the polytope using H-representation with 2D

A2 = np.array([[1, 0], [0, 1], [-1, 0], [0, -1]])
b2 = np.array([1, 2, 1, 2])

P_2D = Polytope(A=A2, b=b2)

# Define the polytope using H-representation with 1D

P_1D = Polytope(A=np.array([[1], [-1]]), b=np.array([3, 3]))

# Define the polytope using H-representation with 3D as Chartesian product of 1D and 2D

P = cartesian_product(P_1D, P_2D)


ax, _, _ = P.plot()
plt.show() 

In [ ]:
F = Polytope(A = P.A/P.b[:,np.newaxis], b= np.ones((len(P.b),1)))
#ax, _, _ = F.plot()
#plt.show

P.A[:,:1],P.A[:,1:]

In [ ]:
# Check proportional scalebility of sets

sigP = 0.5 * P

ax, _, _ = sigP.plot()
plt.show() 


# Casadi Tests

In [ ]:
import numpy as np
import casadi as ca 
from pycvxset import Polytope

In [ ]:
def ca_einsum(a: np.ndarray, b: ca.MX):
    """
    Perform the equivalent of np.einsum('ijk,kl->ijl', a, b) between a 3D np.ndarray and a casadi.MX object.
    Arguments:
    ----------
    a: 3D np.ndarray
        The first input array.
    b: casadi.MX
        The second input array.
    Returns:
    --------
    np.ndarray
        The result of the einsum operation.
    """
    res = np.reshape(a,(a.shape[0]*a.shape[1],b.shape[0])) @ b
    return res if len(b.shape)>1 and b.shape[-1]>1 else ca.reshape(res,a.shape[1],a.shape[0]).T 

### Test `ca_einsum()`

In [ ]:
tensor = np.array([[[1.0, 0.5, 0.25],[3, 1.5, 0.75]], [[1.0, 0.5, 0.25],[3, 1.5, 0.75]], [[1.0, 0.5, 0.25],[3, 1.5, 0.75]]]).transpose(1, 2, 0)
symbolic = ca.MX.sym("M",3, 1)
fun = ca.Function(
    'fun',
    [symbolic],
    [ca_einsum(tensor, symbolic)]
)

array2D = np.array([[1.0], [2.0], [4.0]])

result = fun(array2D)
print(result)

array1D = array2D.squeeze()
result = fun(array1D)

print(result)

valued_matrix = np.array([[1.0, 3.0, 5.0, 7.0], [2.0, 6.0, 10.0, 14.0], [4.0, 12.0, 20.0, 28.0]])

result_valued = fun.map(4)(valued_matrix)
print(result_valued.toarray().reshape(2,4,3)[:,0,:])

camx2D = ca.MX.sym("mx2D", 3, 1)

print(fun(camx2D))

camx1D = ca.MX.sym("mx1D", 3)

print(fun(camx1D))


In [ ]:
cadm = ca.DM(np.array([[1], [2], [3], [4]]))
print(ca.blockcat([[cadm,ca.DM.zeros(4,3)]]))


camx = ca.MX.sym('cadm',4,1)
print(ca.blockcat([[camx,ca.DM.zeros(4,3)]]))

print(ca.horzcat(ca.DM.ones(3,1),ca.DM.zeros(3,3)))


In [ ]:
import numpy as np
a1 = ca.SX.sym('a1',1,1)
a2 = ca.SX.sym('a2',1,1)
a3 = ca.SX.sym('a3',1,1)
a4 = ca.SX.sym('a4',1,1)
a5 = ca.SX.sym('a5',1,1)
a6 = ca.SX.sym('a6',1,1)


mat = ca.blockcat([[a1],[a2]])

print(ca.repmat(mat,1,3)*np.matlib.repmat(np.array(range(1,4)),2,1))


In [ ]:
mat1 = ca.blockcat([[a1],[a2]])

mat2 = ca.blockcat([[a3, a4],[a5, a6]])

print(mat1+mat2)

In [ ]:
o1 = ca.DM.ones(2,1)

print(o1+0.1)

In [ ]:
n_par = 3
n_vert = 2

c = ca.MX.sym('c', n_par, 1)
c_v = ca.MX.sym('c_v', n_par*n_vert, 1)

A = np.array([[[1.0, 1.0],[1.0, 1.0]], [[0.5, 0.5], [0.5, 0.5]], [[0.25, 0.25], [0.25, 0.25]]]).transpose(1, 2, 0)

f = ca.Function(
    'f',
    [c],
    [ca_einsum(A, c)]
)

f_v = ca.Function(
    'f_v',
    [c_v],
    [ca_einsum(A, ca.reshape(c_v,n_vert,n_par).T)]
)

In [ ]:
p_v = np.array([[[ 1], [2]]])
p = np.array([[[ 1]], [[2]], [[4]]])
p_N = np.array([[[ 1, 3, 5, 7]]])

th, th_v, th_N, th_v_N  = p, p*p_v, p*p_N, p*p_v*p_N 

In [ ]:
# th = np.array([[1], [2], [4]])
sym_res = f(th.reshape(-1,1))
arr_res = sym_res.toarray().reshape(2,2,-1).squeeze()


arr_res

In [ ]:
th_v.reshape(-1,1),th_v.squeeze(),th_v.reshape(-1,1).reshape(n_par,n_vert)

In [ ]:
np.einsum('ijk,kl...->ijl...',A,th_v.squeeze())[:,:,0]

In [ ]:
th_v.shape

In [ ]:
# th_v = np.array([[1, 2, 3], [2, 4, 6], [4, 8, 12]])
sym_res = f_v(th_v.squeeze().reshape(-1,1))
arr_res = sym_res.toarray()#.reshape(2,2,-1).squeeze()

sym_res.shape

In [ ]:
sym_res = f.map(3)(th_N.squeeze())
arr_res = sym_res.toarray().reshape(2,2,-1).squeeze()

arr_res[:,:,0]

In [ ]:
# th_v_N = np.array([[1, 2, 3], [2, 4, 6], [4, 8, 12]])*np.array([[[1]], [[5]], [[7]]]).transpose(1,2,0)
# th_v_N.shape
sym_res = f_v.map(4)(th_v_N.reshape(-1,4))
arr_res = sym_res.toarray().reshape(2,2,-1,4).squeeze()

# arr_res[np.array(range(4))*3+0,0].reshape(2,2)
arr_res[:,:,0,0]

In [ ]:
sym_res

In [ ]:
# casadi.Function calling casadi.Function inside
c_g = ca.MX.sym('c_g', n_par, 1)


g = ca.Function(
    'g',
    [c],
    [f(c)]
)

g(th.reshape(-1,1))

g_v = ca.Function(
    'g_v',
    [c_v],
    [f_v(c_v)]
)
 
g_v.map(3)(th_v_N.reshape(-1,3)).toarray().reshape(2,2,-1,3).squeeze()[:,:,0,0]
# f_v.map(3)(th_v_N.reshape(-1,3)).toarray().reshape(2,2,-1,3).squeeze()[:,:,0,0]

# Mapping works also over casadi.Functions

In [ ]:
th_N.shape

In [ ]:
# list comprehension with casadi.Function 
c_N = ca.MX.sym('c_N', n_par, 3)

f_N = ca.Function(
    'f_N',
    [c_N],
    [c_N[:,v] for v in range(3)]
)

f_N(th_N.reshape(-1,3))


In [ ]:
# Variable overwriting test

l = ca.MX.sym('l', 3, 1)
l = ca.MX.sym('l', 3, 6)
l.shape

In [ ]:
b=np.array([1, 2, 3, 4, 5, 6])

if False:
    a = slice(3)
else:
    a = -1

print(b[a])


In [ ]:
N = 30
n = 3
m = 2
op = ca.Opti()

x_opti =op.variable(n)
op.set_initial(x_opti,[1,2,3])
print(x_opti)

Q = ca.MX(np.array([[1,0,0],[0,2,0],[0,0,3]]))

# A = np.random(n,n)
# B = np.random(n.m)

x = ca.MX.sym('x',n,1,N)
x[0]=ca.MX(np.array([1,2,3]))
u = ca.MX.sym('u',m,1,N)
print(Q)
print(Q@x[0])
# constraint = []

# for i in range(N):

#     constraint.append(x[i+1]==A @ x[i] + B @ u[i])
#     ca.qpsol()



## Casadi - Solving a NLP with Opti

In [ ]:
opti = ca.Opti()

x = opti.variable()
y = opti.variable()

opti.minimize((y-x**2)**2)
opti.subject_to(x**2+y**2==1)
opti.subject_to(x+y>1)

opti.solver('ipopt')

sol = opti.solve()

print(sol.value(x))
print(sol.value(y))

## Casadi - Solving a NLP with MX

In [ ]:
x = ca.MX.sym('x')
y = ca.MX.sym('y')
z = ca.MX.sym('z')

F = ca.MX.sym("F")
G = ca.MX.sym("G")
f = (y-x**2)**2
g = ca.vertcat(x**2+y**2,x+y)
nlp = {'x':ca.vertcat(x,y), 'f':f, 'g':g, 'p':z}
s = ca.nlpsol('s', 'ipopt', nlp)
r = s(lbg=ca.vertcat(1,1+z), ubg=ca.vertcat(1,np.abs(-np.inf)),x0=0.786151)
x_opt = r['x']
print(x_opt)


In [ ]:
z = ca.vertcat(x,y)
Const = ca.horzcat(F,G)
Const@z
g = ca.vertcat(g,Const@z)
const_value = []
h = np.array([[1,1,1],[2,2,2],[3,3,3]])
H = ca.MX(h)
print(H)

## Transforming a MX object into a numpy array - need to pass by DM object

In [ ]:
print(H)

g=ca.MX()

g=ca.vertcat(g,H)
g

# CVXPY LMI Solver

In [ ]:
import cvxpy as cp
import numpy as np

import os
import mosek

os.environ["MOSEKLM_LICENSE_FILE"] = "27007@192.168.100.118"
mosek.Env().getversion()
mosek.Env().putlicensepath("27007@192.168.100.118")
mosek.Env().expirylicenses()

# Data
A = np.array([[0, 1], [-1, -2]])
n = 2

# Variables
P = cp.Variable((n, n), symmetric=True)

tol = cp.Parameter(1,"tol")

# Constraints
# P > 0 and A.T*P + P*A < 0 (using a small epsilon for strict inequality)
constraints = [P >> tol * np.eye(n), 
               A.T @ P + P @ A << -tol * np.eye(n)]

# Problem
prob = cp.Problem(cp.Minimize(0), constraints)
#prob.solve()

#print("Status:", prob.status)
#print("P:\n", P.value)


In [ ]:
tol.value = [1e-6]
prob.solve()
print("Status:", prob.status)
print("P:\n", P.value)

In [ ]:
import scipy as sc
l=cp.Parameter(1,"lambda")
print(l)
l.shape

A = np.array([[1,2],[2,1]]).transpose()
print(np.linalg.inv(A))

In [ ]:
rho = np.array([[.9, .1, 2]])

if np.isnan(rho).any():
    print("Hello")

np.isnan(rho).any()

In [ ]:
import numpy as np
A=np.array([[[1, 2],[3, 4]],
            [[5, 6],[7, 8]],
            [[9, 10],[11, 12]]])
len(A.shape)
A_trans = np.copy(A).T

B=np.array([[1, 2],[3, 4]])
len(B.shape)

if len(A.shape)>2:
    v=A.shape
else:
    v=1


if len(B.shape)==2:
    B = B[np.newaxis, :, :]

b = np.array([1, 0])

B @ b

# Example of A[0]*a[0] + A[1]*a[1] + A[2]*a[2] ... where a is a vector of parameters and the first is always 1

a = np.array([0, 1, 0],).T
A = A.transpose(1, 2, 0)

A_trans @ a, A @ a, A[:, :, 0]


## pagemtimes() subsitution

In [ ]:
import numpy as np









# Simple MPC QP with CasADi

## 1st attempt 

In [ ]:
import numpy as np
import casadi as ca
import osqp

x0=np.array([[10],[0]])
u0=np.array([[0]])

# System Matricies

A = ca.MX(np.array([[1, 1],[0, 1]]))
B = ca.MX(np.array([[0], [1]]))
C = ca.MX(np.array([[1,0]]))


N = 2 # Prediction horizon 


x = ca.MX.sym('x',2,N+1)
u = ca.MX.sym('u',1,N)
y = ca.MX.sym('y',1,N)
X = ca.MX.sym('X',2)
U = ca.MX.sym('U',1)
lti=ca.Function('lti',[X,U],[A@X+B@U,C@X],['x_0','u_0'],['x_1','y_0'])

f = 0
Q = 1 
R = 0.1
P = np.eye(2)


for i in range(N):
    f +=y[:,i].T@Q@y[:,i]+u[:,i].T@R@u[:,i]

f += x[:,-1].T@P@x[:,-1]

g=ca.vertcat(x[:,0],ca.reshape(ca.vertcat(
              u, 
             (y - C@x[:,1:]),
             (x[:,1]-A@x[:,:-2]-B@u)),-1,1))

lbg = ca.vertcat(lti(x0,u0),-np.ones((N,1)),np.zeros((N,1)),np.zeros((N*2,1)))
ubg = ca.vertcat(lti(x0,u0),np.ones((N,1)),np.zeros((N,1)),np.zeros((N*2,1)))

qp = {'f':f, 'x':ca.vertcat(ca.reshape(x,-1,1),ca.reshape(u,-1,1),ca.reshape(y,-1,1)), 'g': g}

mysol = ca.qpsol('mysol', 
                 'qpoases', 
                 qp)

display(mysol)
mysol(lbg=lbg, ubg=ubg)




## 2nd attempt - iterative contruction of cost and constraints

In [ ]:
import casadi as ca
import numpy as np
# --- System Setup ---
n_x, n_u, n_y = 2, 1, 1
N = 10 # Longer horizon for testing

# Matrices (use DM for numeric constants, MX for symbolic)
A = ca.DM([[1, 1], [0, 1]])
B = ca.DM([[0], [1]])
C = ca.DM([[1, 0]])
Q, R, P = 1, 0.1, np.eye(2)

# --- Symbolic Formulation ---
w = []      # Decision variables (x0, u0, x1, u1...)
lbw, ubw = [], []
g = []      # Constraints (dynamics and output)
lbg, ubg = [], []
J = 0       # Cost function

# Parameter for initial state (to allow MPC in a loop)
x_init = ca.MX.sym('x_init', n_x)

# Current state (starting from symbolic x0)
Xk = ca.MX.sym('X_0', n_x)
w.append(Xk)
lbw.extend([-ca.inf]*n_x); ubw.extend([ca.inf]*n_x)

# Constraint x0 = x_init
g.append(Xk - x_init)
lbg.extend([0]*n_x); ubg.extend([0]*n_x)

for k in range(N):
    Uk = ca.MX.sym(f'U_{k}', n_u)
    w.append(Uk)
    # Input bounds (very simple here)
    lbw.append(-1.0); ubw.append(1.0)
    
    # Output and stage cost
    Yk = C @ Xk
    J += Yk.T @ Q @ Yk + Uk.T @ R @ Uk
    
    # Dynamics (equality constraint)
    X_next = A @ Xk + B @ Uk
    Xk = ca.MX.sym(f'X_{k+1}', n_x)
    w.append(Xk)
    lbw.extend([-ca.inf]*n_x); ubw.extend([ca.inf]*n_x)
    
    g.append(Xk - X_next)
    lbg.extend([0]*n_x); ubg.extend([0]*n_x)

# Terminal cost
J += Xk.T @ P @ Xk

qp_struct = {'f': J, 'x': ca.vertcat(*w), 'g': ca.vertcat(*g), 'p': x_init}

# Options for OSQP
opts = {'osqp': {'verbose': False}}

# CREATE SOLVER (once)
solver = ca.qpsol('solver', 'osqp', qp_struct, opts)

curr_x=np.array([[10],[0]])
solver(p=curr_x, lbx=lbw, ubx=ubw, lbg=lbg, ubg=ubg)

#u_control = sol['x'][n_x] # Take the first input after state x0


## 3rd attempt - Casadi Function usage for faster evaluation of constraints and costs

In [ ]:
import casadi as ca
import numpy as np

def cost_fun_build(X, U, X_all, U_all, C, P, R, N):
    f_cost = ca.Function('f_cost', [X, U],
                         [X.T @ C.T @ Q @ C @ X + U.T @ R @ U])
    stage_costs = f_cost.map(N)(X_all[:, :N], U_all)
    J = ca.sum2(stage_costs) + X_all[:, N].T @ P @ X_all[:, N]
    return J

def constraint_build(X, U, X_all, U_all, A, B, N, x_init):
    f_step = ca.Function('f_step', [X, U], [ A @ X + B @ U])
    X_next_sim = f_step.map(N)(X_all[:, :N], U_all)
    
    g_dyn = X_all[:, 1:] - X_next_sim
    g_init = X_all[:, 0] - x_init
    g = ca.vertcat(g_init, ca.reshape(g_dyn, -1, 1))
    return g


#Qc = C.T@Q@C

def set_controller(x_init, z, J, g, solver, options={}):
    qp = {'x': z, 'f': J, 'g': g, 'p': x_init}
    return ca.qpsol('qpsol', solver, qp, options)

def solve(qpsol, x0, lbw, ubw, lbg, ubg):
    return qpsol(p=x0, lbx=lbw, ubx=ubw, lbg=lbg, ubg=ubg)

In [ ]:
from pycvxset import Polytope
n_x,n_u, n_y, N = 2, 1, 1, 2

A = ca.DM([[1,1], [0, 1]])
B = ca.DM([[0], [1]])
C = ca.DM([[1, 0]])
Q, R, P = ca.DM(1), ca.DM(0.1), ca.DM(np.eye(2))

Qc = C.T@Q@C

_Q = ca.MX.sym('_Q',n_x,n_x)
_R = ca.MX.sym('_R',n_u,n_u)  
_x = ca.MX.sym('_x',n_x)
_u = ca.MX.sym('_u',n_u)  

quad_cost = ca.Function('quad_cost',[_x,_u,_Q,_R],[_x.T@_Q@_x + _u.T@_R@_u])  

X = ca.MX.sym('x', n_x)
U = ca.MX.sym('u', n_u)
X_all = ca.MX.sym('X_all', n_x, N+1)
U_all = ca.MX.sym('U_all', n_u, N)

x_a = ca.MX.sym('xa', n_x)
X_a = ca.MX.sym('Xa', n_x, N)


state_quad = ca.Function('state_quad', [X],
                             [X.T @ Q @ X])

#output_quad = ca.Function('output_quad', [X, Qc],
#                             [X.T @ Qc @ X])


#input_quad = ca.Function('input_quad', [U, R],
#                             [U.T @ R @ U])

x_init = ca.MX.sym('x_init', n_x)

J = cost_fun_build(X, U, X_all, U_all, C, P, R, N)
g = constraint_build(X, U, X_all, U_all, A, B, N, x_init)

z = ca.vertcat(ca.reshape(X_all, -1, 1), ca.reshape(U_all, -1, 1))

P = Polytope(A=np.array([[1, 0], [0, 1], [-1, 0], [0, -1]]), b=np.array([1, 1, 1, 1]))
F = (P.A/P.b[:, np.newaxis])[:, :n_x*(N+1)]
G = (P.A/P.b[:, np.newaxis])[:, n_x*(N+1):]
F = ca.MX(F)
G = ca.MX(G)

g = ca.vertcat(g, ca.reshape(F, -1, 1), ca.reshape(G, -1, 1))
#z_reshaped=ca.vertcat(ca.reshape(ca.vertcat(X_all[:,:N], U_all),-1,1),X_all[:,N])


qpsol = set_controller(x_init, z, J, g, 'osqp')
x0 = np.array([[10], [0]])

lbx = [-ca.inf]*(n_x*(N+1))
ubx = [ca.inf]*(n_x*(N+1))

lbu = [-1.0]*n_u*N
ubu = [1.0]*n_u*N

lbw = lbx + lbu
ubw = ubx + ubu

n_g = n_x*N + n_x
lbg = [0]*n_g
ubg = [0]*n_g

print(F.shape)

lbg = lbg + [-ca.inf]*(ca.reshape(F,-1, 1).size(1))
ubg = ubg + [1]*(ca.reshape(F,-1, 1).size(1))

sol = solve(qpsol, x0, lbw, ubw, lbg, ubg)

In [ ]:
u_star = sol['x'][(N+1)*n_x : (N+1)*n_x+n_u]
u_star,sol['x']

In [ ]:
import casadi as ca
import numpy as np

# --- System Setup ---
n_x, n_u, n_y = 2, 1, 1
N = 2 # Longer horizon for testing

# Matrices (use DM for numeric constants, MX for symbolic)
A = ca.DM([[1, 1], [0, 1]])
B = ca.DM([[0], [1]])
C = ca.DM([[1, 0]])
Q, R, P = 1, 0.1, np.eye(2)

# Support symbols (local to the function)
X = ca.MX.sym('x', n_x)
U = ca.MX.sym('u', n_u)

# Step function: computes the next state
# (x, u) -> (x_next)
f_step = ca.Function('f_step', [X, U], [A @ X + B @ U])

# Stage cost function: computes the cost for one time step
# (x, u) -> (cost)
f_cost = ca.Function('f_cost', [X, U], [X.T @ C.T @ Q @ C @ X + U.T @ R @ U])

# Decision variables (bulk)
U_all = ca.MX.sym('U', n_u, N)      # Matrix n_u x N (all inputs)
X_all = ca.MX.sym('X', n_x, N+1)    # Matrix n_x x (N+1) (all states)

# --- MAP for Cost ---
# Apply f_cost to each column of X_all and U_all simultaneously
stage_costs = f_cost.map(N)(X_all[:, :N], U_all)
J = ca.sum2(stage_costs) + X_all[:, N].T @ P @ X_all[:, N]

# --- MAP for Dynamics Constraints ---
# Apply f_step to all pairs (x_k, u_k) to obtain all (x_next_k)
X_next_sim = f_step.map(N)(X_all[:, :N], U_all)

# Equality constraints: x_{k+1} - f(x_k, u_k) = 0
g_dyn = X_all[:, 1:] - X_next_sim
# Initial state constraint: x_0 - x_init = 0
g_init = X_all[:, 0] - x_init

g = ca.vertcat(g_init, ca.reshape(g_dyn, -1, 1))


# Single decision vector
w = ca.vertcat(ca.reshape(X_all, -1, 1), ca.reshape(U_all, -1, 1))

qp_struct = {'f': J, 'x': w, 'g': g, 'p': x_init}
solver = ca.qpsol('solver', 'osqp', qp_struct)

# Bounds for X (n_x * (N+1) elements)
lbx = [-ca.inf] * (n_x * (N + 1))
ubx = [ca.inf] * (n_x * (N + 1))

# Bounds for U (n_u * N elements)
lbu = [-1.0] * (n_u * N)  # Example: saturation at -1
ubu = [1.0] * (n_u * N)   # Example: saturation at 1

# Combine variable bounds
lbw = lbx + lbu
ubw = ubx + ubu

# g_init has dimension n_x (x0 - x_init = 0)
# g_dyn has dimension n_x * N (dynamics x_{k+1} - Ax - Bu = 0)
n_constraints = n_x + (n_x * N)

lbg = [0.0] * n_constraints
ubg = [0.0] * n_constraints

# Example of a current initial state
curr_x = np.array([10.0, 0.0])

# Solve
solver(
    p=curr_x,      # Pass the initial state to the x_init parameter
    lbx=lbw,       # Bounds on variables (States and Inputs)
    ubx=ubw, 
    lbg=lbg,       # Bounds on constraints (Dynamics = 0)
    ubg=ubg
)


# Class Debug

## `cMPC()`

In [ ]:
from cMPC import CMPC
import numpy as np
from pycvxset import Polytope

import os
import mosek

from pycvxset import common as cpy
from pycvxset.common import constants

if constants.DEFAULT_LP_SOLVER_STR == 'MOSEK':
    constants.DEFAULT_LP_SOLVER_STR = 'CLARABEL'
    constants.DEFAULT_SOCP_SOLVER_STR = 'CLARABEL'
    constants.DEFAULT_SDP_SOLVER_STR = 'CLARABEL'

# os.environ["MOSEKLM_LICENSE_FILE"] = "/opt/mosek/11.0/mosek.lic"
# mosek.Env().getversion()
# mosek.Env().putlicensepath("/opt/mosek/11.0/mosek.lic")
# mosek.Env().expirylicenses()

A = np.array([[1,1], [0, 1]])
B = np.array([[0], [1]])
C = np.array([[1, 0]])
Q, R = np.eye(2), np.array([[.1]])

K =-np.array([[1.19, 7.88]])
opt = {
    'solver':'osqp',
    'verbose':True,
    'svd':False,
    'xBound':(np.array([-8, -8]), np.array([8, 8])),
    'uBound':(np.array([-1]),np.array([1])),
    'name':'test_mpc',
    'ref': 'traj'
}

ap = np.array([[1.1, 2], [0, 0.95]])
bp = np.array([[0], [0.0787]])
cp = np.array([[-1, 1]])

myMPC = CMPC({'A':A,'B':B,'C':C},Q,R,10,opt)
myMPC.initialize(mode="volume")
curr_x = np.array([2.0, 0.0])
myMPC.solve(curr_x, [0., 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.])
myMPC.u_star



In [ ]:
import matplotlib.pyplot as plt
ax,_,_= myMPC.poly_x_aug.projection([2,3,4]).plot()
plt.plot([0, 1], [0, 1], marker='o')
plt.show()

## `cRMPC()`

In [ ]:
from cRMPC import CRMPC
import numpy as np
from pycvxset import Polytope

# import os
# import mosek

from pycvxset import common as cpy
from pycvxset.common import constants

# if constants.DEFAULT_LP_SOLVER_STR == 'MOSEK':
#     constants.DEFAULT_LP_SOLVER_STR = 'CLARABEL'
#     constants.DEFAULT_SOCP_SOLVER_STR = 'CLARABEL'
#     constants.DEFAULT_SDP_SOLVER_STR = 'CLARABEL'

N = 10

test = "test2"  # "test2" "test1"

if test == "test1":  # From Book
    
    A1 = np.array([[-0.7, 0.15], [-0.35, -0.6]])
    A2 = np.array([[-0.75, -0.1], [0.15, -0.65]])
    A3 = np.array([[-0.65, -0.35], [-0.1, -0.55]])

    A0 = 1/3*(A1+A2+A3)


    dA1 = A1 - A0
    dA2 = A2 - A0
    dA3 = A3 - A0

    B1 = np.array([[0.1], [1]])
    B2 = np.array([[0.2], [1.4]])
    B3 = np.array([[0.3], [0.6]])

    B0 = 1/3*(B1+B2+B3)


    dB1 = B1 - B0
    dB2 = B2 - B0
    dB3 = B3 - B0

    theta_v = np.eye(3)#U[:, :2].T @ blocked_delta
    Theta = Polytope(V=theta_v.T)

    A = np.stack([A0, dA1, dA2, dA3], axis=2)
    B = np.stack([B0, dB1, dB2, dB3], axis=2)


    C = np.array([[1, 0]]) # This is enough
    C = np.stack([C, np.zeros((1, 2)), np.zeros((1, 2)), np.zeros((1, 2))], axis=2) # this is correct and can be used to test if c_mask works properly

    Q, R = np.eye(2), np.array([[1]])
    K =np.array([[0.19, 0.34]])

    opt = {
    'K': K,
    'solver':'osqp',
    'verbose':True,
    'svd':False,
    'xBound':(np.array([-1e16, -10]), np.array([1e16, 10])),
    'uBound':(np.array([-5]),np.array([5])),
    'name':'test_mpc',
    'W': Polytope(A=np.block([[np.eye(2)], [-np.eye(2)]]), b=0.05*np.ones((4,1))),
    'lam': 0.999,
    'theta': Theta,
    }
    curr_x = np.array([2.5, 10.0])

if test == "test2":  # From Lu Cannon  --  Now not working because of polytope Z representation
    A0 = np.array([[0.5, 0.2], [-0.1, 0.6]])
    dA1 = np.array([[0.042, 0.], [0.072, 0.03]])
    dA2 = np.array([[-0.0015, 0.019], [0.009, -0.035]])
    dA3 = np.array([[0., 0.], [0., 0.]])

    B0 = np.array([[0.], [0.5]])
    dB1 = np.array([[0.], [0.]])
    dB2 = np.array([[0.], [0.]])
    dB3 = np.array([[0.0397], [0.059]])

    A = np.stack([A0, dA1, dA2, dA3], axis=2)
    B = np.stack([B0, dB1, dB2, dB3], axis=2)

    C = np.array([[1, 0]])
    C = np.stack([C, np.zeros((1, 2)), np.zeros((1, 2)), np.zeros((1, 2))], axis=2)

    Q, R = np.eye(2), np.array([[1]])
    Theta = Polytope(A = np.block([[np.eye(3)], [-np.eye(3)]]), b=np.ones(6,))
    # from matplotlib import pyplot as plt
    # ax,_,_= Theta.plot()
    # plt.show()  


    opt = {
    'solver':'osqp',
    'verbose':True,
    'svd':False,
    'xBound':(np.array([-1e4, -0.3]), np.array([1e4, 1e4])),
    'uBound':(np.array([-1e4]),np.array([1])),
    'name':'test_mpc',
    'W': Polytope(A=np.block([[np.eye(2)], [-np.eye(2)]]), b=0.05*np.ones((4,1))),
    'lam': 0.95,
    'theta': Theta,
    'ref': True
    }

    # from matplotlib import pyplot as plt
    # ax,_,_= opt['W'].plot()
    # plt.show()  

    
    curr_x = np.array([3, 6])

myMPC = CRMPC({'A':A,'B':B,'C':C},Q,R,N,opt)
# myMPC.lam= 0.9
# myMPC.initialize_uncertainties(theta=Theta)
myMPC.initialize(mode="performance")

# from matplotlib import pyplot as plt
# ax, _, _= myMPC.V.plot()

# ax.grid(True)
# plt.show()  

myMPC.solve(curr_x,np.zeros((1,1)))  # , [0., 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
myMPC.u_star



## `cRAMPC()`


In [1]:
from cRAMPC import CRAMPC
import numpy as np
from pycvxset import Polytope

# import os
# import mosek

from pycvxset import common as cpy
from pycvxset.common import constants

# if constants.DEFAULT_LP_SOLVER_STR == 'MOSEK':
#     constants.DEFAULT_LP_SOLVER_STR = 'CLARABEL'
#     constants.DEFAULT_SOCP_SOLVER_STR = 'CLARABEL'
#     constants.DEFAULT_SDP_SOLVER_STR = 'CLARABEL'

N = 10

test = "test2"  # "test2" "test1"

if test == "test1":  # From Book
    
    A1 = np.array([[-0.7, 0.15], [-0.35, -0.6]])
    A2 = np.array([[-0.75, -0.1], [0.15, -0.65]])
    A3 = np.array([[-0.65, -0.35], [-0.1, -0.55]])

    A0 = 1/3*(A1+A2+A3)


    dA1 = A1 - A0
    dA2 = A2 - A0
    dA3 = A3 - A0

    B1 = np.array([[0.1], [1]])
    B2 = np.array([[0.2], [1.4]])
    B3 = np.array([[0.3], [0.6]])

    B0 = 1/3*(B1+B2+B3)


    dB1 = B1 - B0
    dB2 = B2 - B0
    dB3 = B3 - B0

    theta_v = np.eye(3)#U[:, :2].T @ blocked_delta
    Theta = Polytope(V=theta_v.T)

    A = np.stack([A0, dA1, dA2, dA3], axis=2)
    B = np.stack([B0, dB1, dB2, dB3], axis=2)


    C = np.array([[1, 0]]) # This is enough
    C = np.stack([C, np.zeros((1, 2)), np.zeros((1, 2)), np.zeros((1, 2))], axis=2) # this is correct and can be used to test if c_mask works properly

    Q, R = np.eye(2), np.array([[1]])
    K =np.array([[0.19, 0.34]])

    opt = {
    'K': K,
    'solver':'osqp',
    'verbose':True,
    'svd':False,
    'xBound':(np.array([-1e16, -10]), np.array([1e16, 10])),
    'uBound':(np.array([-5]),np.array([5])),
    'name':'test_mpc',
    'W': Polytope(A=np.block([[np.eye(2)], [-np.eye(2)]]), b=0.05*np.ones((4,1))),
    'lam': 0.999,
    'theta': Theta
    }
    curr_x = np.array([2.5, 10.0])

if test == "test2":  # From Lu Cannon  --  Now not working because of polytope Z representation
    A0 = np.array([[0.5, 0.2], [-0.1, 0.6]])
    dA1 = np.array([[0.042, 0.], [0.072, 0.03]])
    dA2 = np.array([[-0.0015, 0.019], [0.009, -0.035]])
    dA3 = np.array([[0., 0.], [0., 0.]])

    B0 = np.array([[0.], [0.5]])
    dB1 = np.array([[0.], [0.]])
    dB2 = np.array([[0.], [0.]])
    dB3 = np.array([[0.0397], [0.059]])

    A = np.stack([A0, dA1, dA2, dA3], axis=2)
    B = np.stack([B0, dB1, dB2, dB3], axis=2)

    C = np.array([[1, 0]])
    C = np.stack([C, np.zeros((1, 2)), np.zeros((1, 2)), np.zeros((1, 2))], axis=2)

    Q, R = np.eye(2), np.array([[1]])
    # theta_v = np.array([[0.8, 0., 0.], [0., 0.2, 0.], [0., 0., -0.5]])
    Theta = Polytope(A = np.vstack([np.eye(3), -np.eye(3)]), b=0.9*np.ones(6,))
    # from matplotlib import pyplot as plt
    # ax,_,_= Theta.plot()
    # plt.show()  


    opt = {
    'solver':'osqp',
    'verbose':True,
    'svd':False,
    'xBound':(np.array([-1e4, -0.3]), np.array([1e4, 1e4])),
    'uBound':(np.array([-1e4]),np.array([1])),
    'name':'test_mpc',
    'W': Polytope(A=np.block([[np.eye(2)], [-np.eye(2)]]), b=0.05*np.ones((4,1))),
    'lam': 0.999,
    'theta': Theta
    }

    # from matplotlib import pyplot as plt
    # ax,_,_= opt['W'].plot()
    # plt.show()  

    
    curr_x = np.array([3, 6])

myMPC = CRAMPC({'A':A,'B':B,'C':C},Q,R,N,opt)
# myMPC.lam= 0.9
# myMPC.initialize_uncertainties(theta=Theta)
myMPC.initialize(mode="LQR")

# from matplotlib import pyplot as plt
# ax, _, _= myMPC.V.plot()
# ax, _, _= myMPC.poly_x_aug.plot()
# ax.grid(True)
# plt.show()  

myMPC.solve(curr_x, curr_x[0])  # , [0., 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
myMPC.u_star



g:\OneDrive - Politecnico di Torino\Research Material\(q)LPV MPC  - Koopman Operator MPC\Code\RAMPC\adaptive_tool.py:134: UserWarning: Theta_c set is empty
  warn('Theta_c set is empty')


Convergence reached after 1 iterations


g:\OneDrive - Politecnico di Torino\Research Material\(q)LPV MPC  - Koopman Operator MPC\Code\RAMPC\cRMPC.py:861: UserWarning: The problem includes expressions that don't support CPP backend. Defaulting to the SCIPY backend for canonicalization.
  w_lp.solve()


map10_lin_constr(horzcat(x[:20]', u')', repmat([-0, -3.33333, -0], 10)){0}'
map10_lin_constr(horzcat(x[:20]', u')', repmat([-0, -2.18294e-16, 1], 10)){0}'
map10_lin_constr(horzcat(x[:20]', u')', repmat([-0, 0.0001, -0], 10)){0}'
0x1
(x[:2]-x_init)
step(zeros(2x1), 0, (reshape(mac(
[[0.5, 0.042, -0.0015, 0], 
 [0.2, 0, 0.019, 0], 
 [-0.1, 0.072, 0.009, 0], 
 [0.6, 0.03, -0.035, 0]],th,zeros(4x1)))'-
[[1, 0], 
 [0, 1]]), mac(
[[0, 0, 0, 0.0397], 
 [0.5, 0, 0, 0.059]],th,zeros(2x1))){0}
@1=mac(
[[0, 0, 0, 0.0397], 
 [0.5, 0, 0, 0.059]],th,zeros(2x1)), vec((x[2:22]-(map10_step((x[:20]-repmat(zeros(2x1), 10)), c, repmat(reshape(mac(
[[0.5, 0.042, -0.0015, 0.0053097], 
 [0.2, 0, 0.019, -0.0486921], 
 [-0.0331272, 0.072, 0.009, 0.00789099], 
 [-0.0132509, 0.03, -0.035, -0.0723636]],th,zeros(4x1)))', 10), repmat(@1, 10)){0}+repmat(step(zeros(2x1), 0, reshape(mac(
[[0.5, 0.042, -0.0015, 0], 
 [0.2, 0, 0.019, 0], 
 [-0.1, 0.072, 0.009, 0], 
 [0.6, 0.03, -0.035, 0]],th,zeros(4x1)))', @1){0}, 10))

Function qpsol_qpsol (0x245fee455a0)
Input 0 (h): all zero sparse: 192-by-192
Input 1 (g): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Input 2 (a): sparse: 48-by-192, 768 nnz
 (36, 0) -> -1
 (42, 0) -> 0
 (43, 0) -> 0
 (44, 0) -> 0
 (45, 0) -> 0
 (46, 0) -> 0
 (47, 0) -> 0
 (37, 1) -> -1
 (42, 1) -> 0
 (43, 1) -> 0
 (44, 1) -> 0
 (45, 1) -> 0
 (46, 1) -> 0
 (47, 1) -> 0
 (38, 2) -> -1
 (42, 2) -> 0
 (43, 2) -> 0
 (44, 2) -> 0
 (45, 2) -> 0
 (46, 2) -> 0
 (47, 2) 

RuntimeError: Error in Function::call for 'qpsol' [MXFunction] at .../casadi/core/function.cpp:1432:
Error in Function::call for 'qpsol' [MXFunction] at .../casadi/core/function.cpp:361:
Error in Function::operator() for 'qpsol_qpsol' [OsqpInterface] at .../casadi/core/function.cpp:1513:
.../casadi/core/conic.cpp:551: conic process failed. Set 'error_on_fail' option to false to ignore this error.

In [ ]:
import matplotlib.pyplot as plt
ax,_,_= myMPC.poly_x_aug.plot()
# plt.plot([0, 1], [0, 1], marker='o')
plt.show()

# Polytope Tests

In [ ]:
from pycvxset import Polytope
import numpy as np

import os
import mosek

INF = 1e16

os.environ["MOSEKLM_LICENSE_FILE"] = "27000@192.168.100.118"
mosek.Env().getversion()
mosek.Env().putlicensepath("27000@192.168.100.118")
mosek.Env().expirylicenses()

A = np.vstack((np.eye(3),-np.eye(3)))
b = np.array([INF, INF, 1, INF, INF, 1])

P = Polytope(A=A,b=b)
P.minimize_H_rep()
P.A



V = np.array([[INF , -INF , 1],
              [INF , -INF , -1],
              [INF , INF , 1],
              [INF , INF , -1],
              [-INF , INF , 1],
              [-INF , INF , -1],
              [-INF , -INF , 1],
              [-INF , -INF , -1]])

P2=Polytope(V=V)
P2.A



S=P2.projection(2)
P3 = P2.projection(0)
P2.projection(2)
#S.intersection_with_halfspaces(P)

In [ ]:
import numpy as np
from itertools import combinations
from pycvxset import Polytope


class ParametricPolytopeVertices:
    """
    Vertici di {x : A*x <= b} con A fissa e b variabile.

    Il costruttore accetta un Polytope() già calcolato (Set.V viene chiamato
    una sola volta qui) ed estrae la struttura combinatoriale dalle basi attive.
    Le inverse A[I,:]^{-1} vengono precalcolate per aggiornamenti O(V*(n^2+m*n)).
    """

    def __init__(self, polytope: Polytope, tol: float = 1e-8):
        self.tol = tol
        self.A = polytope.A/polytope.b[:, np.newaxis]           # matrice fissa
        self.m, self.n = self.A.shape

        b = np.ones(self.m)             # vettore b iniziale (ones se normalizzato)
        vertices = polytope.V                # unica chiamata costosa, fatta qui

        self.active_bases: list[tuple] = []
        self.basis_inverses: list[np.ndarray] = []

        self._extract_bases(vertices, b)

        self.active_bases = np.array(self.active_bases)
        self.basis_inverses = np.array(self.basis_inverses)

    def _extract_bases(self, vertices: np.ndarray, b: np.ndarray) -> None:
        """
        For each known vertex it found the active basis (n linearly indipendent active constraints) and pre-comput the inverse matrix.
        """
        for v in vertices:
            # Indici dei vincoli attivi per questo vertice: A[i,:]*v ≈ b[i]
            residuals = self.A @ v - b
            active_idx = np.where(np.abs(residuals) < self.tol * 100)[0]

            basis_found = False
            for indices in combinations(active_idx, self.n):
                A_sub = self.A[list(indices), :]
                if abs(np.linalg.det(A_sub)) < self.tol:
                    continue
                self.active_bases.append(indices)
                self.basis_inverses.append(np.linalg.inv(A_sub))
                basis_found = True
                break

            if not basis_found:
                raise RuntimeError(
                    f"No basis found for vertex {v}. "
                    "Augment tolerance."
                )

    def get_vertices(self, b: np.ndarray) -> np.ndarray:
        """
        Computes new vertices {x : A*x <= b_new} from the precomputed bases.

        For each basis I:  v_new = A[I,:]^{-1} @ b_new[I] 
        Check feasibility on all constraints and deduplicate.

        Arguments:
        ----------
            b: new rhs vector, shape (m,) or (m,1) or (m,N)

        Returns:
        --------
            vertices: np.ndarray shape (V, n) or (V, n, N) if b is (m,N)
        """
        vertices = np.einsum("ijk,ik...->ij...", self.basis_inverses, b[self.active_bases])
        
        return vertices.transpose(0, 1, 2).squeeze()

In [ ]:
np.array([[1, 2]])[1:,:]

In [ ]:
from matplotlib import pyplot as plt
np.random.seed(542)
A = np.block([[np.random.rand(3, 2)],[-np.random.rand(3, 2)*10]])*10
b = np.random.rand(6, 1)*18
Set = Polytope(A=A, b=b)

ax,_,_ = Set.plot()
# plt.show()

test = ParametricPolytopeVertices(Set)

b_contr = np.ones((6, 1))*0.5 # Contracted 
b_exp = np.ones((6, 1))*1.5 # Expanded

vertices_contr = test.get_vertices(b_contr)
vertices_exp = test.get_vertices(b_exp)

Set_contr = Polytope(V=vertices_contr)
Set_exp = Polytope(V=vertices_exp)
ax,_,_ = Set_contr.plot(ax=ax, patch_args={'facecolor':'red', 'alpha':0.5})
ax,_,_ = Set_exp.plot(ax=ax, patch_args={'facecolor':'green', 'alpha':0.2})
plt.show()

In [ ]:
from matplotlib import pyplot as plt
np.random.seed(542)
A = np.block([[np.random.rand(3, 2)],[-np.random.rand(3, 2)*10]])*10
b = np.random.rand(6, 1)*18
Set = Polytope(A=A, b=b)

ax,_,_ = Set.plot()
# plt.show()

test = ParametricPolytopeVertices(Set)

b_1 = np.ones((6, 1))*0.5  
b_2 = np.ones((6, 1))*1.5 
b_3 = np.ones((6, 1))*0.7 
b_4 = np.ones((6, 1))*1.2
b_5 = np.ones((6, 1))*0.3

b_new = np.hstack((b_1, b_2, b_3, b_4, b_5))
vertices_new = test.get_vertices(b_new)

for mutations in range(vertices_new.shape[2]):
    
    ax,_,_ = Polytope(V=vertices_new[:,:,mutations]).plot(ax=ax, patch_args={'facecolor':'red', 'alpha':0.15*mutations})

plt.show()

In [ ]:
vertices_new.shape

vertices_new_aug = np.concatenate((np.ones((4,1,5)), vertices_new), axis=1)

vertices_new_aug[:,:,2]

In [ ]:
from pycvxset import Polytope
import numpy as np

def lam_contract_set(A, X0, lam):
    """Compute the lambda-contractive( or invariant) set.

    Parameters
    ----------
    A : np.ndarray
        System matrix, used to evolve the set.
    X0 : Polytope
        Initial polytope.
    lam : float
        Lambda contraction factor (1 for invariant set).
    """

    # Check if it there is more than 1 vertex

    if len(A.shape) > 2:
        v = np.size(A, 2)
    else:
        v = 1

    Xa = X0
    max_iter = 100
    A_tmp = np.eye(np.size(A, 0))

    for k in range(max_iter):

        A_tmp = np.einsum("ik...,kj...->ij...", A_tmp, A / lam)
        Xa_1 = Polytope(
            A=np.vstack(
                ( Xa.A, np.einsum("ij,jk...->ik...", X0.A, A_tmp))
            ).squeeze(),
            b=np.concatenate(( Xa.b, np.matlib.repmat(X0.b, v, 1).squeeze())),
        )
        Xa_1.minimize_H_rep()

        if Xa == Xa_1:
            break
            # TODO check with a test if the algorithm can converge
        Xa = Xa_1
        print("Iteration:", k)
A = np.array([[11, 2], [0, 0.95]])
B = np.array([[0], [0.0787]])

P = Polytope(A=A, b=B)
lam_contract_set(A, P, 0.9)